## **agentic ai - assignment alpha (q3)**

### **23K-0074**

### **dataset**

In [1]:
import numpy as np
import pandas as pd

data = pd.DataFrame({
    "Wind": ["Weak", "Strong", "Weak", "Weak", "Strong"],
    "Humidity": ["High", "High", "Low", "Low", "Low"],
    "Rain": ["Yes", "Yes", "No", "Yes", "No"]
})

classes = ["Yes", "No"]
features = ["Wind", "Humidity"]
values = {"Wind": ["Weak", "Strong"], "Humidity": ["High", "Low"]}

print(data)
print("\nclass counts:")
print(data["Rain"].value_counts())

     Wind Humidity Rain
0    Weak     High  Yes
1  Strong     High  Yes
2    Weak      Low   No
3    Weak      Low  Yes
4  Strong      Low   No

class counts:
Rain
Yes    3
No     2
Name: count, dtype: int64


### **part 1 - naive bayes (laplace smoothing)**

In [2]:
import numpy as np
import pandas as pd

#priors
priors = {}
for cl in classes:
    priors[cl] = (data["Rain"] == cl).sum() / len(data)
    print(f"P(Rain={cl}) = {(data['Rain'] == cl).sum()}/{len(data)} = {priors[cl]:.2f}")

#likelihoods with laplace smoothing: (count + 1) / (class count + k), k = no. of values of the feature
likelihoods = {}
print()
for cl in classes:
    subset = data[data["Rain"] == cl]
    for f in features:
        k = len(values[f])
        for v in values[f]:
            count = (subset[f] == v).sum()
            likelihoods[(f, v, cl)] = (count + 1) / (len(subset) + k)
            print(f"P({f}={v} | Rain={cl}) = ({count}+1)/({len(subset)}+{k}) = {likelihoods[(f, v, cl)]:.4f}")

P(Rain=Yes) = 3/5 = 0.60
P(Rain=No) = 2/5 = 0.40

P(Wind=Weak | Rain=Yes) = (2+1)/(3+2) = 0.6000
P(Wind=Strong | Rain=Yes) = (1+1)/(3+2) = 0.4000
P(Humidity=High | Rain=Yes) = (2+1)/(3+2) = 0.6000
P(Humidity=Low | Rain=Yes) = (1+1)/(3+2) = 0.4000
P(Wind=Weak | Rain=No) = (1+1)/(2+2) = 0.5000
P(Wind=Strong | Rain=No) = (1+1)/(2+2) = 0.5000
P(Humidity=High | Rain=No) = (0+1)/(2+2) = 0.2500
P(Humidity=Low | Rain=No) = (2+1)/(2+2) = 0.7500


In [3]:
#Humidity=High never occurs with Rain=No, without smoothing this would be 0
print("unsmoothed:", 0 / 2)
print("smoothed:", likelihoods[("Humidity", "High", "No")])

unsmoothed: 0.0
smoothed: 0.25


In [4]:
def nb_predict(wind, humidity):
    scores = {}
    for cl in classes:
        scores[cl] = priors[cl] * likelihoods[("Wind", wind, cl)] * likelihoods[("Humidity", humidity, cl)]
    total = sum(scores.values())
    post = {cl: scores[cl] / total for cl in classes}
    return scores, post, max(scores, key=scores.get)

for w in values["Wind"]:
    for h in values["Humidity"]:
        scores, post, pred = nb_predict(w, h)
        print(f"wind={w}, humidity={h} | yes: {scores['Yes']:.4f}  no: {scores['No']:.4f} | P(yes)={post['Yes']:.3f} -> rain={pred}")

wind=Weak, humidity=High | yes: 0.2160  no: 0.0500 | P(yes)=0.812 -> rain=Yes
wind=Weak, humidity=Low | yes: 0.1440  no: 0.1500 | P(yes)=0.490 -> rain=No
wind=Strong, humidity=High | yes: 0.1440  no: 0.0500 | P(yes)=0.742 -> rain=Yes
wind=Strong, humidity=Low | yes: 0.0960  no: 0.1500 | P(yes)=0.390 -> rain=No


In [5]:
from sklearn.naive_bayes import CategoricalNB

#encoding: Wind Strong=1, Humidity High=1, Rain Yes=1
X_enc = pd.DataFrame({
    "Wind": data["Wind"].map({"Weak": 0, "Strong": 1}),
    "Humidity": data["Humidity"].map({"Low": 0, "High": 1})
})
y_enc = data["Rain"].map({"No": 0, "Yes": 1})
all_cases = pd.DataFrame({"Wind": [0, 0, 1, 1], "Humidity": [1, 0, 1, 0]})

#alpha=1 -> laplace smoothing
cnb = CategoricalNB(alpha=1)
cnb.fit(X_enc, y_enc)
print("sklearn P(yes) for (weak,high), (weak,low), (strong,high), (strong,low):")
print(cnb.predict_proba(all_cases)[:, 1].round(3))

sklearn P(yes) for (weak,high), (weak,low), (strong,high), (strong,low):
[0.812 0.49  0.742 0.39 ]


### **part 2 - entropy and information gain**

In [6]:
import numpy as np
import pandas as pd

def entropy(labels):
    probs = labels.value_counts(normalize=True)
    return -np.sum(probs * np.log2(probs)) + 0.0

def info_gain(df, feature, target="Rain"):
    total_h = entropy(df[target])
    weighted = 0
    for v in df[feature].unique():
        sub = df[df[feature] == v]
        h = entropy(sub[target])
        weighted += len(sub) / len(df) * h
        print(f"{feature}={v}: {sub[target].tolist()} -> H = {h:.4f}")
    print(f"weighted entropy = {weighted:.4f}")
    return total_h - weighted

print("H(S) =", round(entropy(data["Rain"]), 4), "\n")
for f in features:
    print(f)
    print(f"IG = {info_gain(data, f):.4f}\n")

H(S) = 0.971 

Wind
Wind=Weak: ['Yes', 'No', 'Yes'] -> H = 0.9183
Wind=Strong: ['Yes', 'No'] -> H = 1.0000
weighted entropy = 0.9510
IG = 0.0200

Humidity
Humidity=High: ['Yes', 'Yes'] -> H = 0.0000
Humidity=Low: ['No', 'Yes', 'No'] -> H = 0.9183
weighted entropy = 0.5510
IG = 0.4200



In [7]:
#humidity has higher IG -> root. High branch is pure (all yes), so split the Low branch on wind
low = data[data["Humidity"] == "Low"]
print("H(low branch) =", round(entropy(low["Rain"]), 4))
print(f"IG = {info_gain(low, 'Wind'):.4f}")

H(low branch) = 0.9183
Wind=Weak: ['No', 'Yes'] -> H = 1.0000
Wind=Strong: ['No'] -> H = 0.0000
weighted entropy = 0.6667
IG = 0.2516


In [9]:
#weak wind + low humidity is a tie (1 yes, 1 no) -> assumed yes, safer for the drone to not fly
def tree_predict(wind, humidity):
    if humidity == "High":
        return "Yes"
    if wind == "Strong":
        return "No"
    return "Yes"

print("humidity?")
print(" |-- high -> rain = yes")
print(" |-- low  -> wind?")
print("             |-- strong -> rain = no")
print("             |-- weak   -> tie, assumed yes\n")

for w in values["Wind"]:
    for h in values["Humidity"]:
        print(f"wind={w}, humidity={h} -> rain={tree_predict(w, h)}")

humidity?
 |-- high -> rain = yes
 |-- low  -> wind?
             |-- strong -> rain = no
             |-- weak   -> tie, assumed yes

wind=Weak, humidity=High -> rain=Yes
wind=Weak, humidity=Low -> rain=Yes
wind=Strong, humidity=High -> rain=Yes
wind=Strong, humidity=Low -> rain=No


In [10]:
from sklearn.tree import DecisionTreeClassifier, export_text

#sklearn breaks the tie by picking the lower class (no)
dt = DecisionTreeClassifier(criterion="entropy", random_state=0)
dt.fit(X_enc, y_enc)
print(export_text(dt, feature_names=["Wind(Strong=1)", "Humidity(High=1)"]))

|--- Humidity(High=1) <= 0.50
|   |--- Wind(Strong=1) <= 0.50
|   |   |--- class: 0
|   |--- Wind(Strong=1) >  0.50
|   |   |--- class: 0
|--- Humidity(High=1) >  0.50
|   |--- class: 1



### **part 3 - logistic regression**

In [11]:
import numpy as np

X = X_enc.values.astype(float)
y = y_enc.values.astype(float)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

#z = b + w1*wind + w2*humidity, trained with gradient descent on binary cross entropy
w = np.zeros(2)
bias = 0.0
lr = 0.5
epochs = 1000
m = len(y)

for epoch in range(epochs):
    p = sigmoid(X @ w + bias)
    loss = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    dw = (X.T @ (p - y)) / m
    db = np.mean(p - y)
    w -= lr * dw
    bias -= lr * db
    if epoch < 2 or epoch % 200 == 0:
        print(f"epoch {epoch}: loss={loss:.4f}, w1={w[0]:.4f}, w2={w[1]:.4f}, b={bias:.4f}")

print(f"\nfinal: w1 (wind) = {w[0]:.4f}, w2 (humidity) = {w[1]:.4f}, b = {bias:.4f}")

epoch 0: loss=0.6931, w1=0.0000, w2=0.1000, b=0.0500
epoch 1: loss=0.6695, w1=-0.0050, w2=0.1925, b=0.0888
epoch 200: loss=0.3211, w1=-1.9128, w2=4.5153, b=-0.0740
epoch 400: loss=0.3007, w1=-2.5855, w2=5.7463, b=-0.0448
epoch 600: loss=0.2932, w1=-2.9894, w2=6.5010, b=-0.0311
epoch 800: loss=0.2894, w1=-3.2771, w2=7.0481, b=-0.0238

final: w1 (wind) = -3.4995, w2 (humidity) = 7.4756, b = -0.0193


In [12]:
#first update by hand: all weights 0 -> every p = 0.5
p0 = sigmoid(np.zeros(m))
print("p - y =", p0 - y)
print("dw =", (X.T @ (p0 - y)) / m, " db =", np.mean(p0 - y))

p - y = [-0.5 -0.5  0.5 -0.5  0.5]
dw = [ 0.  -0.2]  db = -0.1


In [13]:
def lr_predict(wind, humidity):
    x = np.array([1.0 if wind == "Strong" else 0.0, 1.0 if humidity == "High" else 0.0])
    p = sigmoid(x @ w + bias)
    return p, "Yes" if p >= 0.5 else "No"

for wv in values["Wind"]:
    for h in values["Humidity"]:
        p, pred = lr_predict(wv, h)
        print(f"wind={wv}, humidity={h} -> P(rain)={p:.3f} -> rain={pred}")

wind=Weak, humidity=High -> P(rain)=0.999 -> rain=Yes
wind=Weak, humidity=Low -> P(rain)=0.495 -> rain=No
wind=Strong, humidity=High -> P(rain)=0.981 -> rain=Yes
wind=Strong, humidity=Low -> P(rain)=0.029 -> rain=No


In [14]:
from sklearn.linear_model import LogisticRegression

#sklearn uses L2 regularization by default (C=1), on 5 samples this shrinks the weights a lot
lr_sk = LogisticRegression()
lr_sk.fit(X_enc, y_enc)
print("w =", lr_sk.coef_.round(4), " b =", lr_sk.intercept_.round(4))
print("P(rain) for all cases:", lr_sk.predict_proba(all_cases)[:, 1].round(3))

w = [[-0.1792  0.6309]]  b = [0.235]
P(rain) for all cases: [0.704 0.558 0.665 0.514]


### **part 4 - comparison**

In [15]:
import numpy as np
import pandas as pd

rows = []
for wv in values["Wind"]:
    for h in values["Humidity"]:
        _, post, nb_pred = nb_predict(wv, h)
        p, lr_pred = lr_predict(wv, h)
        rows.append([wv, h, round(post["Yes"], 3), nb_pred, tree_predict(wv, h), round(p, 3), lr_pred])

compare = pd.DataFrame(rows, columns=["Wind", "Humidity", "NB P(Yes)", "NB", "Tree", "LR P(Yes)", "LR"])

#drone delivers only if the majority predicts no rain
majority = (compare[["NB", "Tree", "LR"]] == "Yes").sum(axis=1) >= 2
compare["Deliver?"] = np.where(majority, "No", "Yes")
print(compare.to_string(index=False))

  Wind Humidity  NB P(Yes)  NB Tree  LR P(Yes)  LR Deliver?
  Weak     High      0.812 Yes  Yes      0.999 Yes       No
  Weak      Low      0.490  No  Yes      0.495  No      Yes
Strong     High      0.742 Yes  Yes      0.981 Yes       No
Strong      Low      0.390  No   No      0.029  No      Yes


In [16]:
acc = {
    "naive bayes": np.mean([nb_predict(r.Wind, r.Humidity)[2] == r.Rain for r in data.itertuples()]),
    "decision tree": np.mean([tree_predict(r.Wind, r.Humidity) == r.Rain for r in data.itertuples()]),
    "logistic regression": np.mean([lr_predict(r.Wind, r.Humidity)[1] == r.Rain for r in data.itertuples()])
}
print("training accuracy:")
for k, v in acc.items():
    print(f"  {k}: {v:.0%}")

training accuracy:
  naive bayes: 80%
  decision tree: 80%
  logistic regression: 80%


| | naive bayes | entropy / IG (tree) | logistic regression |
|---|---|---|---|
| type | probabilistic, generative | rule based, splits on features | probabilistic, discriminative |
| assumption | features independent given class | none, greedy splits | log odds linear in features |
| output | class probabilities | class label (rules) | class probability |
| small data | works well, smoothing handles zero counts | ties and overfitting | weights keep growing if a feature separates a class |
| interpretability | likelihood tables | easiest, readable rules | weights show feature effect |

- humidity is the stronger predictor in all three (IG 0.42 vs 0.02, LR weight 7.48 vs -3.50)
- all three agree on high humidity -> rain and strong wind + low humidity -> no rain
- only disagreement is weak wind + low humidity, the data has one yes and one no for it, NB says no by a small margin (0.144 vs 0.150), tree tie assumed yes
- all models get 80% training accuracy since rows 3 and 4 have the same features but different labels
- humidity = high always gives yes, so unregularized LR keeps increasing w2, the fixed epochs stop it. sklearn's L2 version predicts yes everywhere, so LR is sensitive to regularization on such small data
- 5 samples is too little for real deployment